In [1]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)

hand_count = 0
hand_inside = False

while True:
    success, frame = cap.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    # box position
    x1, y1 = 200, 100
    x2, y2 = 500, 400

    # box ke andar ka area
    roi = frame[y1:y2, x1:x2]

    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    lower_skin = np.array([0, 30, 60])
    upper_skin = np.array([25, 255, 255])

    mask = cv2.inRange(hsv, lower_skin, upper_skin)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=2)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    hand_detected = False

    if len(contours) > 0:
        hand = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(hand)

        if area > 5000:
            hand_detected = True
            cv2.drawContours(roi, [hand], -1, (0, 255, 0), 2)

    if hand_detected and hand_inside == False:
        hand_count += 1
        hand_inside = True

    if hand_detected == False:
        hand_inside = False

    count_text = f"Hand Count : {hand_count}"

    cv2.rectangle(
        frame,
        (0, 0),
        (330, 50),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        frame,
        count_text,
        (10, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    # main rectangle box
    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        (255, 0, 0),
        3
    )

    cv2.imshow("Hand Movement Counter", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

print("Final hand movement count:", hand_count)

cap.release()
cv2.destroyAllWindows()

Final hand movement count: 58
